<a href="https://colab.research.google.com/github/kmillaevelyn/data-science-portfolio/blob/main/PI3_Dashboard_Vagas_LinkedIn_Skills.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %%capture
# Instalação das bibliotecas necessárias
!pip install pandas openpyxl matplotlib seaborn plotly streamlit

import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os
import subprocess
import time
import textwrap

# --- Definições de keywords para referência
hard_skills_keywords_global = ['python', 'sql', 'java', 'c++', 'javascript', 'excel', 'power bi', 'tableau', 'aws', 'azure', 'data science', 'machine learning', 'cloud', 'segurança cibernética']
soft_skills_keywords_global = ['comunicação', 'resolução de problemas', 'trabalho em equipe', 'pensamento crítico', 'adaptabilidade', 'liderança', 'proatividade', 'criatividade', 'organização']
ferramentas_keywords_global = ['excel', 'power bi', 'tableau', 'jira', 'trello', 'github', 'vscode', 'docker', 'kubernetes', 'salesforce', 'sap']

# --- Geração do arquivo streamlit_app.py - TOTALMENTE AUTOCONTIDO ---
app_code_content = textwrap.dedent("""
import streamlit as st
import pandas as pd
import plotly.express as px
import os
import ast

st.set_page_config(layout="wide", page_title="BCTec: Desvendando Vagas do Futuro")

# --- FUNÇÕES E DADOS ---

# Caminho do arquivo CSV
csv_file_path_app = 'Vagas Linkedin - Pagina1.csv'

@st.cache_data
def load_data_app(file_path):
    if not os.path.exists(file_path):
        st.error(f"Erro: Arquivo CSV '{file_path}' não encontrado. Certifique-se de que ele está no mesmo diretório do app Streamlit.")
        return pd.DataFrame()
    try:
        df_app = pd.read_csv(file_path)
        df_app.columns = df_app.columns.str.lower().str.replace(' ', '_').str.replace('[^a-zA-Z0-9_]', '', regex=True)
        return df_app
    except Exception as e:
        st.error(f"Erro ao carregar o arquivo CSV no app: {e}")
        return pd.DataFrame()

df_app = load_data_app(csv_file_path_app)

# Definições de keywords
hard_skills_keywords_app = ['python', 'sql', 'java', 'c++', 'javascript', 'excel', 'power bi', 'tableau', 'aws', 'azure', 'data science', 'machine learning', 'cloud', 'segurança cibernética']
soft_skills_keywords_app = ['comunicação', 'resolução de problemas', 'trabalho em equipe', 'pensamento crítico', 'adaptabilidade', 'liderança', 'proatividade', 'criatividade', 'organização']
ferramentas_keywords_app = ['excel', 'power bi', 'tableau', 'jira', 'trello', 'github', 'vscode', 'docker', 'kubernetes', 'salesforce', 'sap']

def extract_keywords_app(text_series, keywords_list):
    flat_list = []
    if text_series is None or text_series.empty:
        return []
    for text in text_series.dropna():
        if pd.isna(text):
            continue
        text_lower = str(text).lower()
        for keyword in keywords_list:
            if keyword.lower() in text_lower:
                flat_list.append(keyword)
    return flat_list

# Processamento dos dados
if df_app.empty:
    st.warning("Não foi possível carregar os dados para o dashboard. Exibindo dados simulados.")
    tecnologia_counts_final = pd.DataFrame({'Área': ['Tecnologia', 'Outras'], 'Número de Vagas': [800, 200]})
    hard_skills_counts_final = pd.DataFrame({'Habilidade': ['Python', 'SQL', 'Java'], 'Contagem': [500, 450, 300]})
    soft_skills_counts_final = pd.DataFrame({'Habilidade': ['Comunicação', 'Resolução de Problemas'], 'Contagem': [150, 120]})
    ferramentas_counts_final = pd.DataFrame({'Ferramenta': ['Excel', 'Power BI'], 'Contagem': [400, 350]})
else:
    # Processamento para dados reais
    if 'area_tecnologica' not in df_app.columns:
        if 'titulo_da_vaga' in df_app.columns:
            df_app['area_tecnologica'] = df_app['titulo_da_vaga'].apply(
                lambda x: 'Tecnologia' if any(term in str(x).lower() for term in ['engenheiro', 'desenvolvedor', 'data', 'analista', 'tech']) else 'Outras')
        else:
            df_app['area_tecnologica'] = 'Tecnologia'

    if 'requisitos' not in df_app.columns:
        if 'descricao_da_vaga' in df_app.columns:
            df_app['requisitos'] = df_app['descricao_da_vaga']
        else:
            df_app['requisitos'] = ""

    all_skills_app = extract_keywords_app(df_app['requisitos'], hard_skills_keywords_app + soft_skills_keywords_app)
    soft_skills_extracted_app = extract_keywords_app(df_app['requisitos'], soft_skills_keywords_app)
    ferramentas_extracted_app = extract_keywords_app(df_app['requisitos'], ferramentas_keywords_app)

    tecnologia_counts_final = df_app['area_tecnologica'].value_counts().reset_index()
    tecnologia_counts_final.columns = ['Área', 'Número de Vagas']

    hard_skills_counts_final = pd.Series([s for s in all_skills_app if s.lower() in [k.lower() for k in hard_skills_keywords_app]]).value_counts().reset_index()
    hard_skills_counts_final.columns = ['Habilidade', 'Contagem']

    soft_skills_counts_final = pd.Series(soft_skills_extracted_app).value_counts().reset_index()
    soft_skills_counts_final.columns = ['Habilidade', 'Contagem']

    ferramentas_counts_final = pd.Series(ferramentas_extracted_app).value_counts().reset_index()
    ferramentas_counts_final.columns = ['Ferramenta', 'Contagem']

# --- CONTEÚDO DO DASHBOARD ---
st.title("📊 BCTec: A Formação Que o Mercado de Trabalho Exige!")
st.markdown("Com base em uma análise aprofundada de milhares de vagas no LinkedIn, desvendamos as principais tendências e habilidades mais procuradas.")

col1, col2 = st.columns(2)

with col1:
    st.header("📈 Demanda por Formação em Tecnologia")
    fig1 = px.bar(tecnologia_counts_final, x='Área', y='Número de Vagas',
                  title='Vagas que Requerem Formação Tecnológica vs. Outras',
                  color='Área', color_discrete_map={'Tecnologia': 'royalblue', 'Outras': 'lightgray'},
                  text='Número de Vagas')
    fig1.update_traces(texttemplate='%{text}', textposition='outside')
    fig1.update_layout(xaxis_title="", yaxis_title="Número de Vagas", showlegend=False)
    st.plotly_chart(fig1, use_container_width=True)
    st.markdown("🚀 **Insight:** A vasta maioria das oportunidades hoje exige uma base tecnológica. O BCTec te dá essa base sólida e versátil!")

with col2:
    st.header("💡 Habilidades Mais Requisitadas (Hard Skills)")
    if not hard_skills_counts_final.empty:
        fig2 = px.bar(hard_skills_counts_final.head(10), x='Contagem', y='Habilidade', orientation='h',
                      title='Top 10 Hard Skills Mais Procuradas',
                      color='Contagem', color_continuous_scale=px.colors.sequential.Viridis)
        fig2.update_layout(yaxis={'categoryorder':'total ascending'}, xaxis_title="Frequência", yaxis_title="")
        st.plotly_chart(fig2, use_container_width=True)
        st.markdown("🎯 **Insight:** O BCTec te capacita nas hard skills essenciais para o mercado, tornando você um profissional tecnicamente preparado.")
    else:
        st.info("Dados de Hard Skills não disponíveis ou insuficientes para análise.")

col3, col4 = st.columns(2)

with col3:
    st.header("🤝 Soft Skills Essenciais")
    if not soft_skills_counts_final.empty:
        fig3 = px.pie(soft_skills_counts_final.head(5), values='Contagem', names='Habilidade',
                      title='Soft Skills Mais Valorizadas',
                      hole=.3)
        fig3.update_traces(textposition='inside', textinfo='percent+label')
        st.plotly_chart(fig3, use_container_width=True)
        st.markdown("🗣️ **Insight:** O mercado valoriza a colaboração, o pensamento crítico e a comunicação. O BCTec desenvolve essas competências através de projetos e desafios práticos.")
    else:
        st.info("Dados de Soft Skills não disponíveis ou insuficientes para análise.")

with col4:
    st.header("🛠️ Ferramentas Que Você Precisa Dominar")
    if not ferramentas_counts_final.empty:
        fig4 = px.bar(ferramentas_counts_final.head(10), x='Ferramenta', y='Contagem',
                      title='Ferramentas Mais Mencionadas nas Vagas',
                      color='Contagem', color_continuous_scale=px.colors.sequential.Plasma)
        fig4.update_layout(xaxis_title="", yaxis_title="Frequência")
        st.plotly_chart(fig4, use_container_width=True)
        st.markdown("⚙️ **Insight:** Esteja à frente com o conhecimento das ferramentas que impulsionam o mercado. O BCTec te guia para dominar as mais relevantes.")
    else:
        st.info("Dados de Ferramentas não disponíveis ou insuficientes para análise.")

st.markdown("---")
st.markdown(
    '''
    ### ✨ BCTec na Unifei: A Escolha Inteligente para o Seu Futuro!
    O Bacharelado em Ciência e Tecnologia da Unifei, lançado em **2024**, é a formação que o mercado de trabalho *realmente* precisa.
    Com uma grade flexível e multidisciplinar, você estará preparado(a) para:
    - **Dominar a tecnologia** e suas aplicações.
    - **Desenvolver habilidades** técnicas e comportamentais essenciais.
    - **Adaptar-se** às rápidas mudanças do mercado.
    - **Construir uma carreira** de sucesso e impactar o mundo!
    '''
)
st.markdown("#### **Venha fazer parte da UNIFEI e seja o profissional do futuro!**")
""")

# Escreve o conteúdo no arquivo
with open("streamlit_app.py", "w", encoding='utf-8') as f:
    f.write(app_code_content)

# Encerra processos existentes na porta 8501
!fuser -k 8501/tcp 2>/dev/null || true

# Executa o Streamlit em uma nova porta
print("Iniciando Streamlit na porta 8502...")
!streamlit run streamlit_app.py --server.port 8502 --server.enableCORS false --browser.gatherUsageStats false

  3687Iniciando Streamlit na porta 8502...
2025-07-03 15:52:24.607 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.61.59.235:8502

